# Long schedule, two cards: w64s24lab211, seeds 0, 1

K=32, all 1.211M training jets, 60 epochs, beams + channels 8 + --keep-best --pair-lab --act-hidden 64 --msg-dim 24 --scalar-dim 24, compiled,
one seed per T4 in parallel. T4 with torch 2.11.0+cu128 pinned and asserted.
Read unpaired against pairlab211 (the same command without the arm's own
flags) and base211, seeds 0 and 1 each.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import os, subprocess, sys

def launch(args, gpu, log):
    """Start a repo module on one card; its output goes to `log`."""
    # unbuffered, with faulthandler: a killed or crashed child still leaves
    # its last lines and a traceback in `log`
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, "-m"] + args, cwd="/kaggle/working/repo",
                            env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(log, n=3000):
    return open(log).read()[-n:]

cards = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True).stdout.splitlines() if l.strip()]
NGPU = len(cards)
print("cards:", cards)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG, SEEDS, EPOCHS = 'w64s24lab211', [0, 1], 60
OUT = "/kaggle/working/results_long_seeds/" + TAG
CKPT = "/kaggle/working/checkpoints/" + TAG
LOGS = "/kaggle/working/logs/" + TAG
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--epochs', '60', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '32', '--eval-chunk-size', '2048', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile', '--keep-best', '--pair-lab', '--act-hidden', '64', '--msg-dim', '24', '--scalar-dim', '24']
BASE[BASE.index("DATA")] = DATA
BUDGET = 11.2 * 3600          # the whole session, minus a margin under 12 h
EVAL_MARGIN = 900             # final test evaluation and writing results
for d in (OUT, LOGS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('a67a3b5b9573' + chr(10))
print("data:", DATA, "| bundle", 'a67a3b5b9573')

In [ ]:
# On-card guard: the forward pass must be the published function on this
# GPU too, in float64. "1 passed" is required; a skip means no CUDA.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1500:], g.stderr[-1500:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
# Smoke on both cards at once, as the real runs will start, each card with
# its own seed, so both seeds' initial weights go through the first-batch
# compile check (on 20k jets' normalisation, not the full set's): one epoch
# on 20k jets per card. Each must write a result.
import glob, json, time
smoke = []
for i in range(min(NGPU, len(SEEDS))):
    d = "/kaggle/working/smoke/%s/card%d" % (TAG, i)
    smoke.append((d, launch(["benchmarks.run_top_tagging"] + BASE +
                            ["--epochs", "1", "--max-train-samples", "20000",
                             "--seed", str(SEEDS[i]), "--results-dir", d],
                            i, LOGS + "/smoke_card%d.log" % i)))
codes = [p.wait() for _, p in smoke]
works = []
for i, ((d, _), rc) in enumerate(zip(smoke, codes)):
    found = glob.glob(d + "/*.json")
    print("card %d: exit %d, result %s" % (i, rc, bool(found)))
    if found:
        r = json.load(open(found[0]))
        print("   seed %d: %.0f s, AUC %.4f, %d params, compile %s, %s"
              % (r["seed"], r["walltime_sec"], r["test_metrics"]["test_auc"], r["n_params"],
                 r.get("compile_check"), r["env"]))
        works.append(i)
    else:
        print(tail(LOGS + "/smoke_card%d.log" % i))
assert works, "the smoke run failed on every card"
PARALLEL = len(works) >= len(SEEDS)
print("parallel:" if PARALLEL else "only card(s) %s work: seed %d alone" % (works, SEEDS[0]),
      "seeds", SEEDS if PARALLEL else SEEDS[:1])

In [ ]:
import glob, time
todo = SEEDS if PARALLEL else SEEDS[:1]
left = int(BUDGET - (time.time() - T0) - EVAL_MARGIN)
print("%.2f h of session budget left for each run" % (left / 3600), flush=True)
assert left > 3600, "not enough session left"
procs = []
for i, s in enumerate(todo):
    gpu = i if PARALLEL else works[0]
    procs.append((s, launch(["benchmarks.run_top_tagging"] + BASE +
                            ["--seed", str(s), "--results-dir", OUT + "/seed%d" % s,
                             "--ckpt-dir", CKPT + "/seed%d" % s,
                             "--resume", "--max-seconds", str(left)],
                            gpu, LOGS + "/seed%d.log" % s)))
    print("seed %d on card %d" % (s, gpu), flush=True)
BAD = []
for s, p in procs:
    rc = p.wait()
    print("=== seed %d: exit %d ===" % (s, rc))
    print(tail(LOGS + "/seed%d.log" % s, 2000))
    if rc != 0 or not glob.glob(OUT + "/seed%d/*.json" % s):
        BAD.append(s)
# a failed seed is reported after the summary below, so the other seed's
# result is still printed
print("failed seeds:", BAD or "none")
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
# Every result against the same-stack long runs: pairlab211 (this command
# without the arm's own flags) and base211, seeds 0 and 1 each. Same-seed
# runs in different kernels do not reproduce on this stack, so the reading
# is unpaired, by the rule fixed before the push (final weights): pays if
# the mean gains >= +0.0003 AUC on pairlab211 and every run lies above both
# pairlab211 runs; close if it gains <= +0.0000; otherwise unresolved. The two
# arm runs share a kernel, so a per-kernel offset would move both.
import json, glob, math, statistics
REFS, RULE = {'pairlab211': [{'seed': 0, 'auc': 0.9831008732725728, 'rej30': 1057.1413612565445, 'val_loss': 0.16442953050136566, 'gap': 0.004935375170900647, 'hours': 3.2990987700925003, 'params': 23026, 'order': '4956a97c5cb4356c52e25711c90c47b09d1f3aa680704b0f679fb5adca1bb0fe'}, {'seed': 1, 'auc': 0.9829955419134161, 'rej30': 934.787037037037, 'val_loss': 0.1646638661623001, 'gap': 0.004633039032989644, 'hours': 3.3460409969541667, 'params': 23026, 'order': '99fdd6358841957e8d3d1294d0b55f55527b4b711eccacdafa8445ab04e904ae'}], 'base211': [{'seed': 0, 'auc': 0.982253273888472, 'rej30': 909.5225225225225, 'val_loss': 0.16791385412216187, 'gap': 0.004503886009325775, 'hours': 3.254759436368889, 'params': 22834, 'order': '4956a97c5cb4356c52e25711c90c47b09d1f3aa680704b0f679fb5adca1bb0fe'}, {'seed': 1, 'auc': 0.9825408318393293, 'rej30': 975.4299516908212, 'val_loss': 0.16674810647964478, 'gap': 0.004954167103196461, 'hours': 3.345618815262222, 'params': 22834, 'order': '99fdd6358841957e8d3d1294d0b55f55527b4b711eccacdafa8445ab04e904ae'}]}, (0.0003, 0.0)
POOLED_SD, POOLED_DF = (0.00021413108438859388, 6)   # within-configuration sd of the AUC at this schedule
mean = statistics.mean
done = []
for f in sorted(glob.glob(OUT + "/seed*/*.json")):
    r = json.load(open(f)); t = r["test_metrics"]; env = r["env"]; bo = r["batch_order"]
    h = r["history"]; vl = [e["val_loss"] for e in h]
    print("seed %d: %d epochs, %.2f h (%.0f s/epoch), %d params | torch %s on %s | compile %s"
          % (r["seed"], r["epochs_run"], r["walltime_sec"] / 3600,
             r["walltime_sec"] / max(r["epochs_run"], 1), r["n_params"], env["torch"], env.get("gpu"),
             r.get("compile_check")))
    assert env["torch"] == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
    ref = {x["seed"]: x for x in REFS["pairlab211"]}.get(r["seed"])
    print("  batch order epoch 1 %s: %s pairlab211's at this seed"
          % (bo["epoch1"][:12], "same as" if ref and ref["order"] == bo["epoch1"] else "DIFFERENT from"))
    print("  AUC %.5f  rej@0.3 %.1f  rej@0.5 %.1f  final val loss %.5f (lowest %.5f at epoch %d)  val - train %+.5f"
          % (t["test_auc"], t["bg_rej_30"], t.get("bg_rej_50", float("nan")), vl[-1],
             min(vl), h[vl.index(min(vl))]["epoch"], vl[-1] - h[-1]["train_loss"]))
    if r.get("test_metrics_best_val"):
        tb = r["test_metrics_best_val"]
        print("  best-val weights (epoch %d): AUC %.5f, rej@0.3 %.1f"
              % (r["best_val_epoch"], tb["test_auc"], tb["bg_rej_30"]))
    if r["epochs_run"] < EPOCHS:
        print("  INCOMPLETE (epoch %d of %d): stopped on the session budget; finish before reading"
              % (r["epochs_run"], EPOCHS))
        continue
    done.append({"auc": t["test_auc"], "rej30": t["bg_rej_30"], "val_loss": vl[-1],
                 "gap": vl[-1] - h[-1]["train_loss"]})
for name, rows in REFS.items():
    print("%-11s AUC %s | rej@0.3 %s | final val loss %s | val - train %s | %s h"
          % (name, " ".join("%.5f" % x["auc"] for x in rows),
             " ".join("%.1f" % x["rej30"] for x in rows),
             " ".join("%.5f" % x["val_loss"] for x in rows),
             " ".join("%+.5f" % x["gap"] for x in rows),
             " ".join("%.2f" % x["hours"] for x in rows)))

def p_two_sided(t, df):
    try:
        from scipy import stats
        return 2 * stats.t.sf(abs(t), df)
    except Exception:
        return float("nan")

def welch(a, b):
    va, vb = statistics.variance(a) / len(a), statistics.variance(b) / len(b)
    t = (mean(a) - mean(b)) / math.sqrt(va + vb)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    return t, df, p_two_sided(t, df)

def pooled(a, b):
    # two runs' sample sd can be far from this schedule's spread (pairlab211's
    # two runs differ by only 0.0001), so also against the pooled sd
    t = (mean(a) - mean(b)) / (POOLED_SD * math.sqrt(1 / len(a) + 1 / len(b)))
    return t, POOLED_DF, p_two_sided(t, POOLED_DF)

if done:
    a = [x["auc"] for x in done]
    for name, rows in REFS.items():
        b = [x["auc"] for x in rows]
        line = "%s (%d runs) - %s: dAUC %+.5f, d rej@0.3 %+.1f, d final val loss %+.5f, d val - train %+.5f" % (
            TAG, len(done), name, mean(a) - mean(b),
            mean(x["rej30"] for x in done) - mean(x["rej30"] for x in rows),
            mean(x["val_loss"] for x in done) - mean(x["val_loss"] for x in rows),
            mean(x["gap"] for x in done) - mean(x["gap"] for x in rows))
        if len(a) > 1:
            line += " | Welch t %.2f, df %.1f, p %.3f" % welch(a, b)
        line += " | against the pooled sd %.5f: t %.2f, df %d, p %.3f" % ((POOLED_SD,) + pooled(a, b))
        print(line)
    ref = [x["auc"] for x in REFS["pairlab211"]]
    d = mean(a) - mean(ref)
    if len(done) < len(SEEDS):
        verdict = "no verdict: %d of %d runs complete" % (len(done), len(SEEDS))
    elif d >= RULE[0] and min(a) > max(ref):
        verdict = "PAYS: confirm at K=64"
    elif d <= RULE[1]:
        verdict = "CLOSE"
    else:
        verdict = "unresolved"
    print("verdict (%s against pairlab211, dAUC %+.5f, runs above both pairlab211 runs: %d of %d): %s"
          % (TAG, d, sum(x > max(ref) for x in a), len(a), verdict))
assert not BAD, "seeds failed (exit code or no result): %s" % BAD